<font size=10>**DECODING FIRM INFLUENCE**</font>

**Following Sturm et al. (2025), Section 5**

<font color='#BFD72F' size=5>**QUESTION**:</font> *How does network position correlate with earnings in public procurement?*

*"Firms occupying more central positions in the network tend to obtain higher earnings, highlighting the premium associated with strategic network placement."*

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>
- [1. Imports & Data Loading](#1-imports)
- [2. Centrality Measures](#2-centrality)
- [3. Firm-Level Outcomes & Controls](#3-outcomes)
- [4. Regression Analysis](#4-regression)
- [5. Robustness Checks](#5-robustness)
- [6. Visualization](#6-visualization)

# <font color='#BFD72F' size=6>**1. Imports & Data Loading**</font> <a class="anchor" id="1-imports"></a>
[Back to TOC](#toc)

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('Libraries loaded.')

In [ ]:
# Load data
G = nx.read_graphml('../graphs/cobidding_network.graphml')
contracts = pd.read_csv('../data/contracts_clean.csv')
bids_table = pd.read_csv('../data/bids_table.csv')
active_core = pd.read_csv('../data/active_core_firms.csv')
communities = pd.read_csv('../data/community_assignments.csv')

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Contracts: {len(contracts):,}')
print(f'Bids: {len(bids_table):,}')

# <font color='#BFD72F' size=6>**2. Centrality Measures**</font> <a class="anchor" id="2-centrality"></a>
[Back to TOC](#toc)

Paper: *"We employ three network centrality metrics: degree centrality, betweenness centrality, and eigenvector centrality."*

- **Degree centrality**: competitive position (how many firms you co-bid with)
- **Betweenness centrality**: strategic positioning (bridging different market segments)
- **Eigenvector centrality**: competing alongside other influential firms

*"To facilitate comparison, we performed a MinMax scaling transformation."*

In [ ]:
# Compute centrality measures
print('Computing centrality measures...')
deg_cent = nx.degree_centrality(G)
print('  Degree centrality ✓')

bet_cent = nx.betweenness_centrality(G, weight='weight', normalized=True)
print('  Betweenness centrality ✓')

eig_cent = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
print('  Eigenvector centrality ✓')

# Create centrality DataFrame
centrality_df = pd.DataFrame({
    'firm_nif': list(G.nodes()),
    'degree_cent': [deg_cent[n] for n in G.nodes()],
    'betweenness_cent': [bet_cent[n] for n in G.nodes()],
    'eigenvector_cent': [eig_cent[n] for n in G.nodes()]
})

# MinMax scaling (paper requirement)
scaler = MinMaxScaler()
centrality_df[['degree_scaled', 'betweenness_scaled', 'eigenvector_scaled']] = scaler.fit_transform(
    centrality_df[['degree_cent', 'betweenness_cent', 'eigenvector_cent']]
)

print(f'\nCentrality statistics:')
print(centrality_df[['degree_cent', 'betweenness_cent', 'eigenvector_cent']].describe())

In [ ]:
# Correlation between centrality measures
print('Correlation between centrality measures:')
corr = centrality_df[['degree_cent', 'betweenness_cent', 'eigenvector_cent']].corr()
print(corr.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.3f')
ax.set_title('Centrality Correlations')
plt.tight_layout()
plt.show()

# <font color='#BFD72F' size=6>**3. Firm-Level Outcomes & Controls**</font> <a class="anchor" id="3-outcomes"></a>
[Back to TOC](#toc)

Paper Eq. 2 specifies the dependent variable and controls:
- **DV**: log(Earnings per Bid) — *"normalizes the outcome by firm activity and tender opportunities"*
- **Controls**: Winning Rate, Geographic/Service/Entity Specialization (HHI), log(n_contracts)

In [ ]:
# Compute firm-level outcomes
# 1. Earnings per bid = total won amount / total bids
firm_earnings = contracts.groupby('winner_nif')['contract_value'].sum().reset_index()
firm_earnings.columns = ['firm_nif', 'total_earnings']

firm_bids_count = bids_table.groupby('firm_nif').size().reset_index(name='total_bids')

# 2. Winning rate = contracts won / contracts bid on
firm_wins = contracts.groupby('winner_nif').size().reset_index(name='contracts_won')
firm_wins.columns = ['firm_nif', 'contracts_won']

# Merge
firm_data = centrality_df[['firm_nif', 'degree_cent', 'betweenness_cent', 'eigenvector_cent',
                            'degree_scaled', 'betweenness_scaled', 'eigenvector_scaled']].copy()
firm_data = firm_data.merge(firm_earnings, on='firm_nif', how='left')
firm_data = firm_data.merge(firm_bids_count, on='firm_nif', how='left')
firm_data = firm_data.merge(firm_wins, on='firm_nif', how='left')
firm_data = firm_data.merge(communities, on='firm_nif', how='left')

# Fill NaN (firms that never won)
firm_data['total_earnings'] = firm_data['total_earnings'].fillna(0)
firm_data['contracts_won'] = firm_data['contracts_won'].fillna(0)

# Compute derived variables
firm_data['earnings_per_bid'] = firm_data['total_earnings'] / firm_data['total_bids']
firm_data['winning_rate'] = firm_data['contracts_won'] / firm_data['total_bids']

print(f'Firms with data: {len(firm_data):,}')
print(f'Firms with positive earnings: {(firm_data["total_earnings"] > 0).sum():,}')

In [ ]:
# Compute HHI specialization indices
# For each firm: share of activity in each category, HHI = sum(share^2)

def compute_hhi(group_col, bids_df, contracts_df):
    """Compute HHI for each firm based on their bidding distribution across a category."""
    # Merge bids with contract attributes
    bids_with_attr = bids_df.merge(
        contracts_df[['idcontrato', group_col]].drop_duplicates(), 
        on='idcontrato', how='left'
    )
    
    # For each firm, compute share in each category
    firm_cat_counts = bids_with_attr.groupby(['firm_nif', group_col]).size().reset_index(name='count')
    firm_totals = firm_cat_counts.groupby('firm_nif')['count'].sum().reset_index(name='total')
    firm_cat_counts = firm_cat_counts.merge(firm_totals, on='firm_nif')
    firm_cat_counts['share'] = firm_cat_counts['count'] / firm_cat_counts['total']
    firm_cat_counts['share_sq'] = firm_cat_counts['share'] ** 2
    
    # HHI per firm
    hhi = firm_cat_counts.groupby('firm_nif')['share_sq'].sum().reset_index()
    hhi.columns = ['firm_nif', f'hhi_{group_col}']
    return hhi

# Geographic HHI (by city)
hhi_geo = compute_hhi('city', bids_table, contracts)
print(f'Geographic HHI computed for {len(hhi_geo)} firms')

# Service HHI (by CPV sector)
hhi_service = compute_hhi('cpv_sector', bids_table, contracts)
print(f'Service HHI computed for {len(hhi_service)} firms')

# Entity HHI (by adjudicante)
hhi_entity = compute_hhi('adjudicante_nif', bids_table, contracts)
print(f'Entity HHI computed for {len(hhi_entity)} firms')

# Merge HHIs
firm_data = firm_data.merge(hhi_geo, on='firm_nif', how='left')
firm_data = firm_data.merge(hhi_service, on='firm_nif', how='left')
firm_data = firm_data.merge(hhi_entity, on='firm_nif', how='left')
firm_data = firm_data.fillna(1.0)  # Solo firms have HHI=1

In [ ]:
# Prepare regression variables
# Filter to firms with positive earnings (can take log)
reg_data = firm_data[firm_data['earnings_per_bid'] > 0].copy()

# Log transformations
reg_data['log_earnings_per_bid'] = np.log(reg_data['earnings_per_bid'])
reg_data['log_degree'] = np.log(reg_data['degree_cent'] + 1e-10)
reg_data['log_betweenness'] = np.log(reg_data['betweenness_cent'] + 1e-10)
reg_data['log_eigenvector'] = np.log(reg_data['eigenvector_cent'] + 1e-10)
reg_data['log_n'] = np.log(reg_data['total_bids'])

print(f'Regression sample: {len(reg_data):,} firms')
print(f'\nDependent variable stats:')
print(reg_data['log_earnings_per_bid'].describe())

# <font color='#BFD72F' size=6>**4. Regression Analysis**</font> <a class="anchor" id="4-regression"></a>
[Back to TOC](#toc)

Paper Table 2: Three models, each with one centrality measure + controls.

$\\log(Earnings/Bid)_i = \\beta_1 \\cdot centrality_i + \\beta_2 \\cdot WinRate_i + \\beta_3 \\cdot \\log(n_i) + \\beta_4 \\cdot HHI^{geo}_i + \\beta_5 \\cdot HHI^{service}_i + \\beta_6 \\cdot HHI^{entity}_i + \\varepsilon_i$

In [ ]:
# Define control variables
controls = ['winning_rate', 'log_n', 'hhi_city', 'hhi_cpv_sector', 'hhi_adjudicante_nif']

# Model 1: Degree Centrality
X1 = reg_data[['log_degree'] + controls]
X1 = sm.add_constant(X1)
y = reg_data['log_earnings_per_bid']
model1 = sm.OLS(y, X1).fit(cov_type='HC3')  # robust standard errors

# Model 2: Betweenness Centrality
X2 = reg_data[['log_betweenness'] + controls]
X2 = sm.add_constant(X2)
model2 = sm.OLS(y, X2).fit(cov_type='HC3')

# Model 3: Eigenvector Centrality
X3 = reg_data[['log_eigenvector'] + controls]
X3 = sm.add_constant(X3)
model3 = sm.OLS(y, X3).fit(cov_type='HC3')

# Model 4: Base (no centrality)
X4 = reg_data[controls]
X4 = sm.add_constant(X4)
model4 = sm.OLS(y, X4).fit(cov_type='HC3')

print('All 4 models estimated successfully.')

In [ ]:
# Display results in paper's Table 2 format
def format_coef(model, var):
    """Format coefficient with significance stars."""
    if var not in model.params.index:
        return ''
    coef = model.params[var]
    se = model.bse[var]
    pval = model.pvalues[var]
    stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    return f'{coef:.3f}{stars} ({se:.3f})'

print('='*80)
print('TABLE 2: TENDER EARNINGS PER BID MODELS')
print('(Following Sturm et al. 2025, Table 2)')
print('='*80)
print(f'{"Dependent variable:":30s} log(Earnings per Bid)')
print(f'{"":30s} {"(1) Degree":15s} {"(2) Between.":15s} {"(3) Eigenvec.":15s} {"(4) Base":15s}')
print('-'*80)

# Centrality coefficients
print(f'{"log(Degree Cent.)":30s} {format_coef(model1, "log_degree"):15s}')
print(f'{"log(Betweenness Cent.)":30s} {"":15s} {format_coef(model2, "log_betweenness"):15s}')
print(f'{"log(Eigenvector Cent.)":30s} {"":15s} {"":15s} {format_coef(model3, "log_eigenvector"):15s}')
print('-'*80)

# Control variables
for ctrl in controls:
    row = f'{ctrl:30s}'
    for m in [model1, model2, model3, model4]:
        row += f' {format_coef(m, ctrl):15s}'
    print(row)

print('-'*80)
print(f'{"Observations":30s} {model1.nobs:.0f}{"":11s} {model2.nobs:.0f}{"":11s} {model3.nobs:.0f}{"":11s} {model4.nobs:.0f}')
print(f'{"R²":30s} {model1.rsquared:.3f}{"":11s} {model2.rsquared:.3f}{"":11s} {model3.rsquared:.3f}{"":11s} {model4.rsquared:.3f}')
print('='*80)
print('Significance: * p<0.1, ** p<0.05, *** p<0.01 | Robust (HC3) standard errors')

# <font color='#BFD72F' size=6>**5. Robustness Checks**</font> <a class="anchor" id="5-robustness"></a>
[Back to TOC](#toc)

Paper: *"We performed robustness checks by re-estimating the regression for different firm sizes — top 50% and bottom 50%."*

In [ ]:
# Split by firm size (median total earnings)
median_earnings = reg_data['total_earnings'].median()
large_firms = reg_data[reg_data['total_earnings'] >= median_earnings]
small_firms = reg_data[reg_data['total_earnings'] < median_earnings]

print(f'Large firms (top 50%): {len(large_firms)}')
print(f'Small firms (bottom 50%): {len(small_firms)}')

# Re-run for large firms
X_large = large_firms[['log_eigenvector'] + controls]
X_large = sm.add_constant(X_large)
model_large = sm.OLS(large_firms['log_earnings_per_bid'], X_large).fit(cov_type='HC3')

# Re-run for small firms
X_small = small_firms[['log_eigenvector'] + controls]
X_small = sm.add_constant(X_small)
model_small = sm.OLS(small_firms['log_earnings_per_bid'], X_small).fit(cov_type='HC3')

print(f'\nEigenvector centrality coefficient:')
print(f'  Large firms: {model_large.params["log_eigenvector"]:.3f} (p={model_large.pvalues["log_eigenvector"]:.4f})')
print(f'  Small firms: {model_small.params["log_eigenvector"]:.3f} (p={model_small.pvalues["log_eigenvector"]:.4f})')
print(f'\nPaper finding: "Network measures have higher coefficients for smaller firms"')

In [ ]:
# VIF check for multicollinearity
X_vif = reg_data[['log_degree'] + controls]
X_vif = sm.add_constant(X_vif)
vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print('Variance Inflation Factors:')
print(vif_data[vif_data['Variable'] != 'const'].to_string(index=False))
print('\n(VIF > 10 indicates concerning multicollinearity)')

# <font color='#BFD72F' size=6>**6. Visualization**</font> <a class="anchor" id="6-visualization"></a>
[Back to TOC](#toc)

In [ ]:
# Scatter plots: earnings vs centrality
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (cent_col, title) in zip(axes, [
    ('log_degree', 'Degree Centrality'),
    ('log_betweenness', 'Betweenness Centrality'),
    ('log_eigenvector', 'Eigenvector Centrality')
]):
    ax.scatter(reg_data[cent_col], reg_data['log_earnings_per_bid'], 
              alpha=0.3, s=15, color='steelblue')
    # Add regression line
    z = np.polyfit(reg_data[cent_col].values, reg_data['log_earnings_per_bid'].values, 1)
    p = np.poly1d(z)
    x_line = np.linspace(reg_data[cent_col].min(), reg_data[cent_col].max(), 100)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2)
    ax.set_xlabel(f'log({title})', fontsize=10)
    ax.set_ylabel('log(Earnings per Bid)', fontsize=10)
    ax.set_title(title, fontsize=12)

plt.tight_layout()
plt.savefig('../figures/centrality_vs_earnings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Coefficient comparison bar chart
coefs = {
    'Degree': model1.params.get('log_degree', 0),
    'Betweenness': model2.params.get('log_betweenness', 0),
    'Eigenvector': model3.params.get('log_eigenvector', 0)
}
errors = {
    'Degree': model1.bse.get('log_degree', 0),
    'Betweenness': model2.bse.get('log_betweenness', 0),
    'Eigenvector': model3.bse.get('log_eigenvector', 0)
}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(coefs.keys(), coefs.values(), yerr=errors.values(), 
              capsize=5, color=['#636EFA', '#EF553B', '#00CC96'], alpha=0.8)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Coefficient (effect on log earnings/bid)', fontsize=11)
ax.set_title('Impact of Network Centrality on Firm Earnings', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/coefficient_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey Finding: Higher centrality → Higher earnings per bid')
print('This confirms the paper\'s result that network position matters for firm success.')